In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [22]:
df = pd.read_csv(r"/content/Covertype.csv")


X_raw = df.iloc[:, :-1].values
y_raw = df.iloc[:, -1].values

print("Input features shape (X):", X_raw.shape)
print("Target classes shape (y):", y_raw.shape)

Input features shape (X): (581012, 54)
Target classes shape (y): (581012,)


In [23]:
y_zero = y_raw - 1

num_samples = len(y_zero)
Y_onehot = np.zeros((num_samples, 7))

for i in range(num_samples):
    class_label = y_zero[i]
    Y_onehot[i, class_label] = 1

print("First 3 target rows in one-hot format:")
print(Y_onehot[:3])

First 3 target rows in one-hot format:
[[0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1.]]


In [24]:
X_train_raw, X_temp, Y_train, Y_temp, y_train_labels, y_temp_labels = train_test_split(
    X_raw, Y_onehot, y_zero, test_size=0.20, stratify=y_zero
)

X_val_raw, X_test_raw, Y_val, Y_test, y_val_labels, y_test_labels = train_test_split(
    X_temp, Y_temp, y_temp_labels, test_size=0.50, stratify=y_temp_labels
)

print("Training rows:", len(X_train_raw))
print("Validation rows:", len(X_val_raw))
print("Testing rows:", len(X_test_raw))

Training rows: 464809
Validation rows: 58101
Testing rows: 58102


In [25]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

In [26]:
N = len(y_train_labels)
K = 7

class_weights = np.zeros(K)

y_train_true_classes = np.argmax(Y_train, axis=1)
unique_classes, class_counts = np.unique(y_train_true_classes, return_counts=True)


for cls, count in zip(unique_classes, class_counts):
    class_weights[cls] = N / (K * count)

print("Unique classes present:", unique_classes)
print("Class counts:", class_counts)
print("Correct class weights:", class_weights)

Unique classes present: [0 6]
Class counts: [  7000 457809]
Correct class weights: [9.48589796 0.         0.         0.         0.         0.
 0.14504146]


In [27]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0) * 1.0

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

In [28]:
W1 = np.random.randn(54, 64) * 0.1
b1 = np.zeros((1, 64))

W2 = np.random.randn(64, 32) * 0.1
b2 = np.zeros((1, 32))

W3 = np.random.randn(32, 16) * 0.1
b3 = np.zeros((1, 16))

W4 = np.random.randn(16, 7) * 0.1
b4 = np.zeros((1, 7))

In [29]:
def forward_pass(X):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)

    Z3 = np.dot(A2, W3) + b3
    A3 = relu(Z3)

    Z4 = np.dot(A3, W4) + b4
    A4 = softmax(Z4)

    return Z1, A1, Z2, A2, Z3, A3, Z4, A4

In [30]:
def compute_loss(Y_hat, Y_true):
    epsilon = 1e-15
    Y_hat = np.clip(Y_hat, epsilon, 1 - epsilon)

    true_labels = np.argmax(Y_true, axis=1)
    weights_for_samples = class_weights[true_labels]

    log_probs = np.log(Y_hat)
    sample_losses = -np.sum(Y_true * log_probs, axis=1)

    total_loss = np.mean(weights_for_samples * sample_losses)
    return total_loss

In [ ]:
learning_rate = 0.01
batch_size = 128
epochs = 30
num_samples = len(X_train)

train_losses = []

for epoch in range(epochs):
    for i in range(0, num_samples, batch_size):
        X_batch = X_train[i:i+batch_size]
        Y_batch = Y_train[i:i+batch_size]
        B = len(X_batch)

        # Forward pass
        Z1, A1, Z2, A2, Z3, A3, Z4, A4 = forward_pass(X_batch)

        # Calculate gradients
        batch_true_labels = np.argmax(Y_batch, axis=1)
        w_diag = class_weights[batch_true_labels].reshape(-1, 1)

        dZ4 = (w_diag * (A4 - Y_batch)) / B
        dW4 = np.dot(A3.T, dZ4)
        db4 = np.sum(dZ4, axis=0, keepdims=True)

        dA3 = np.dot(dZ4, W4.T)
        dZ3 = dA3 * relu_derivative(Z3)
        dW3 = np.dot(A2.T, dZ3)
        db3 = np.sum(dZ3, axis=0, keepdims=True)

        dA2 = np.dot(dZ3, W3.T)
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = np.dot(X_batch.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update weights and biases
        W4 -= learning_rate * dW4
        b4 -= learning_rate * db4
        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    _, _, _, _, _, _, _, A4_train = forward_pass(X_train[:5000])
    current_loss = compute_loss(A4_train, Y_train[:5000])
    train_losses.append(current_loss)

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {current_loss:.4f}")

Epoch 1/30 - Loss: 0.0292
Epoch 2/30 - Loss: 0.0026
Epoch 3/30 - Loss: 0.0009
Epoch 4/30 - Loss: 0.0005
Epoch 5/30 - Loss: 0.0003
Epoch 6/30 - Loss: 0.0003
Epoch 7/30 - Loss: 0.0002


In [ ]:
plt.plot(train_losses)
plt.title("Training Loss Trajectory")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

In [ ]:
_, _, _, _, _, _, _, test_probs = forward_pass(X_test)

predictions = np.argmax(test_probs, axis=1)

correct_predictions = np.sum(predictions == y_test_labels)
total_test_samples = len(y_test_labels)

accuracy = (correct_predictions / total_test_samples) * 100

print(f"Test Accuracy: {accuracy:.2f}%")